# Scrapping para os textos

In [5]:
import requests
from bs4 import BeautifulSoup

# Assuming 'html_doc' is a variable holding the URL from the kernel state
html_doc = "https://ipip.ori.org/AlphabeticalItemList.htm"
response = requests.get(html_doc)
soup = BeautifulSoup(response.text, 'html.parser')

In [6]:
print(soup.prettify())

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN">
<html xmlns="http://www.w3.org/TR/REC-html40" xmlns:m="http://schemas.microsoft.com/office/2004/12/omml" xmlns:o="urn:schemas-microsoft-com:office:office" xmlns:v="urn:schemas-microsoft-com:vml" xmlns:w="urn:schemas-microsoft-com:office:word">
 <head>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="Word.Document" name="ProgId"/>
  <meta content="Microsoft Word 12" name="Generator"/>
  <meta content="Microsoft Word 12" name="Originator"/>
  <link href="AlphabeticalItemList_files/filelist.xml" rel="File-List"/>
  <title>
   3320 IPIP Items
  </title>
  <link href="AlphabeticalItemList_files/themedata.thmx" rel="themeData"/>
  <link href="AlphabeticalItemList_files/colorschememapping.xml" rel="colorSchemeMapping"/>
  <!--[if gte mso 9]><xml> <w:WordDocument> <w:Zoom>106</w:Zoom> <w:GrammarState>Clean</w:GrammarState> <w:TrackMoves>false</w:TrackMoves> <w:TrackFormatting/> <w:ValidateAga

In [12]:
# Localiza a tabela pela classe específica
tabela = soup.find('table', class_='MsoNormalTable')

if tabela:
    frases_extraidas = []
    # Percorre cada linha da tabela
    for tr in tabela.find_all('tr'):
        # Pega apenas o primeiro td de cada linha
        primeiro_td = tr.find('td')
        if primeiro_td:
            texto = primeiro_td.get_text(strip=True)
            # Filtra ruídos comuns de formatação e códigos de referência
            if texto and len(texto) > 3:
                frases_extraidas.append(texto)

    # Salva as frases em um arquivo TXT
    with open('frases.txt', 'w', encoding='utf-8') as f:
        for frase in frases_extraidas:
            f.write(frase + '\n')

    print(f"Sucesso! {len(frases_extraidas)} frases extraídas.")
    print("Arquivo 'frases.txt' gerado com sucesso.")

    # Exibe as primeiras 5 apenas para conferência
    print("\nExemplo das primeiras frases:")
    for f in frases_extraidas[:5]:
        print(f"- {f}")
else:
    print("Tabela 'MsoNormalTable' não encontrada.")

Sucesso! 3324 frases extraídas.
Arquivo 'frases.txt' gerado com sucesso.

Exemplo das primeiras frases:
- The 3,320 IPIP Items in AlphabeticalOrder(plus their Survey and Item Number1)
- 1All but the 654 T items, 127 W
items, and 1 Y item from the 3,320 IPIP items have been administered to
the Eugene-Springfield Community Sample in 12 of the 30 separate
surveys described in theJanuary 2016 ORI
Technical Report. The survey numbers and letter codes of IPIP
items included in the survey are as follows: (2) H; (8) X; (10) E; (14)
N, B; (15) P; (21) A, C; (24) D; (25) Q; (26) R; (27) M; (28) V, (29)
S. The T items are from the CAT Personality Disorder Scales (Simms, et
al, 2011), the W items are new items added to the VIA measure of
character strengths (du Plessis & de Bruin, 2015), and the Y
item was created for the Big-Five Aspect Scales (DeYoung, Quilty,
& Peterson, 2007).
- Abuse people's
confidences.
- Accept
apologies easily.
- Accept
challenging tasks.


In [18]:
# Limpa as quebras de linha internas de cada frase
frases_limpas = [frase.replace('\r', '').replace('\n', ' ').strip() for frase in frases_extraidas]

# Remove espaços duplos que podem ter surgido na substituição
frases_limpas = [' '.join(frase.split()) for frase in frases_limpas]

# Salva novamente no arquivo TXT com a formatação corrigida
with open('frases.txt', 'w', encoding='utf-8') as f:
    for frase in frases_limpas:
        if frase:
            f.write(frase + '\n')

print(f"Arquivo atualizado! {len(frases_limpas)} frases formatadas.")

# Exibe o exemplo corrigido
print("\nExemplo de como as frases ficaram agora:")
for f in frases_limpas[2:7]: # Pula o cabeçalho longo e mostra os itens
    print(f"- {f}")

Arquivo atualizado! 3324 frases formatadas.

Exemplo de como as frases ficaram agora:
- Abuse people's confidences.
- Accept apologies easily.
- Accept challenging tasks.
- Accept little from others.
- Accept others' weaknesses.


In [7]:
# Carrega as frases do arquivo TXT para uma lista Python
with open('/frases.txt', 'r', encoding='utf-8') as f:
    lista_frases = [linha.strip() for linha in f if linha.strip()]

# Remove os dois primeiros itens (cabeçalho e notas de rodapé da página)
if len(lista_frases) > 2:
    lista_frases = lista_frases[2:]

print(f"Foram carregadas {len(lista_frases)} frases de itens para a variável 'lista_frases'.")

# Exibe as primeiras 5 frases da lista limpa
display(lista_frases[:5])

Foram carregadas 3318 frases de itens para a variável 'lista_frases'.


['Accept challenging tasks.',
 'Accept little from others.',
 "Accept others' weaknesses.",
 'Accept people as they are.',
 'Accept the consequences of my actions.']

# Carregando o modelo para fazer o embeddings

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Carrega o modelo
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
# Gera os embeddings
embeddings_frases = model.encode(
    lista_frases,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Quantidade de frases:", len(lista_frases))
print("Dimensão dos embeddings:", embeddings_frases.shape)

Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Quantidade de frases: 3318
Dimensão dos embeddings: (3318, 768)


In [10]:
Lista_Surgency_positive = ["Extraverted", "Talkative", "Assertive", "Verbal", "Energetic", "Bold", "Active", "Daring", "Vigorous", "Unrestrained"]
Lista_Surgency_negative = ["Introverted", "Shy", "Quiet", "Reserved", "Untalkative", "Inhibited", "Withdrawn", "Timid", "Bashful", "Unadventurous"]

Lista_Agreeableness_positive = ["Kind", "Cooperative", "Sympathetic", "Warm", "Trustful", "Considerate", "Pleasant", "Agreeable", "Helpful", "Generous"]
Lista_Agreeableness_negative = ["Cold", "Unkind", "Unsympathetic", "Distrustful", "Harsh", "Demanding", "Rude", "Selfish", "Uncooperative", "Uncharitable"]

Lista_Conscientiousness_positive = ["Organized", "Systematic", "Thorough", "Practical", "Neat", "Efficient", "Careful", "Steady", "Conscientious", "Prompt"]
Lista_Conscientiousness_negative = ["Disorganized", "Careless", "Unsystematic", "Inefficient", "Undependable", "Unpractical", "Negligent", "Haphazard", "Sloppy", "Unreliable"]

Lista_Emotional_Stability_positive = ["Unevvious", "Unemotional", "Relaxed", "Imperturbable", "Unexcitable", "Undemanding"]
Lista_Emotional_Stability_negative = ["Anxious", "Moody", "Temperamental", "Envious", "Emotional", "Irritable", "Fretful", "Jealous", "Touchy", "Nervous", "Insecure", "Fearful", "Self-pitying", "High-strung"]

Lista_Intellect_positive = ["Intellectual", "Creative", "Complex", "Imaginative", "Bright", "Philosophical", "Artistic", "Deep", "Innovative", "Introspective"]
Lista_Intellect_negative = ["Unintellectual", "Unintelligent", "Unimaginative", "Simple", "Unrefined", "Unsophisticated", "Imperceptive", "Uninquisitive", "Shallow"]

In [11]:
# Gerando embeddings para cada lista de traços de personalidade
emb_Surgency_pos = model.encode(Lista_Surgency_positive, convert_to_numpy=True)
emb_Surgency_neg = model.encode(Lista_Surgency_negative, convert_to_numpy=True)

emb_Agreeableness_pos = model.encode(Lista_Agreeableness_positive, convert_to_numpy=True)
emb_Agreeableness_neg = model.encode(Lista_Agreeableness_negative, convert_to_numpy=True)

emb_Conscientiousness_pos = model.encode(Lista_Conscientiousness_positive, convert_to_numpy=True)
emb_Conscientiousness_neg = model.encode(Lista_Conscientiousness_negative, convert_to_numpy=True)

emb_Emotional_Stability_pos = model.encode(Lista_Emotional_Stability_positive, convert_to_numpy=True)
emb_Emotional_Stability_neg = model.encode(Lista_Emotional_Stability_negative, convert_to_numpy=True)

emb_Intellect_pos = model.encode(Lista_Intellect_positive, convert_to_numpy=True)
emb_Intellect_neg = model.encode(Lista_Intellect_negative, convert_to_numpy=True)

print("Embeddings gerados com sucesso para todas as listas!")
print(f"Exemplo - Dimensão Surgency Positive: {emb_Surgency_pos.shape}")

Embeddings gerados com sucesso para todas as listas!
Exemplo - Dimensão Surgency Positive: (10, 768)


# Similaridade e clusterização

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1. Preparar os dicionários de categorias e calcular as médias (centróides)
categorias = {
    'Surgency+': emb_Surgency_pos.mean(axis=0).reshape(1, -1),
    'Surgency-': emb_Surgency_neg.mean(axis=0).reshape(1, -1),
    'Agreeableness+': emb_Agreeableness_pos.mean(axis=0).reshape(1, -1),
    'Agreeableness-': emb_Agreeableness_neg.mean(axis=0).reshape(1, -1),
    'Conscientiousness+': emb_Conscientiousness_pos.mean(axis=0).reshape(1, -1),
    'Conscientiousness-': emb_Conscientiousness_neg.mean(axis=0).reshape(1, -1),
    'Emotional_Stability+': emb_Emotional_Stability_pos.mean(axis=0).reshape(1, -1),
    'Emotional_Stability-': emb_Emotional_Stability_neg.mean(axis=0).reshape(1, -1),
    'Intellect+': emb_Intellect_pos.mean(axis=0).reshape(1, -1),
    'Intellect-': emb_Intellect_neg.mean(axis=0).reshape(1, -1)
}

nomes_categorias = list(categorias.keys())
centroides = np.vstack(list(categorias.values()))

# 2. Calcular similaridade de cosseno entre todas as frases e os centróides
similaridades = cosine_similarity(embeddings_frases, centroides)

# 3. Identificar a categoria mais próxima para cada frase
indices_max = np.argmax(similaridades, axis=1)
scores_max = np.max(similaridades, axis=1)

# Criar um DataFrame para facilitar a visualização
df_agrupado = pd.DataFrame({
    'Frase': lista_frases,
    'Categoria_Atribuida': [nomes_categorias[i] for i in indices_max],
    'Score_Similaridade': scores_max
})

print("Agrupamento concluído!")
display(df_agrupado.head(10))

Agrupamento concluído!


,Frase,Categoria_Atribuida,Score_Similaridade
0,Accept challenging tasks.,Surgency+,0.475983
1,Accept little from others.,Agreeableness-,0.584581
2,Accept others' weaknesses.,Agreeableness-,0.459585
3,Accept people as they are.,Agreeableness+,0.450564
4,Accept the consequences of my actions.,Agreeableness+,0.395066
5,Accept the first thing that comes along.,Agreeableness+,0.442512
6,Accept what others say.,Agreeableness+,0.534846
7,Accomplish a lot of work.,Conscientiousness+,0.465653
8,Accomplish my work on time.,Conscientiousness+,0.448555
9,Acknowledge others' accomplishments.,Agreeableness+,0.501229


In [13]:
# Exibir a contagem de frases por categoria
contagem = df_agrupado['Categoria_Atribuida'].value_counts()
print("\nDistribuição das frases por categoria:")
print(contagem)

# Mostrar exemplos aleatórios de alguns grupos
print("\nExemplos de frases no grupo 'Intellect+':")
display(df_agrupado[df_agrupado['Categoria_Atribuida'] == 'Intellect+']['Frase'].head(5))

print("\nExemplos de frases no grupo 'Agreeableness+':")
display(df_agrupado[df_agrupado['Categoria_Atribuida'] == 'Agreeableness+']['Frase'].head(5))


Distribuição das frases por categoria:
Categoria_Atribuida
Emotional_Stability-    839
Agreeableness+          513
Agreeableness-          445
Surgency+               327
Conscientiousness+      295
Surgency-               289
Emotional_Stability+    203
Conscientiousness-      165
Intellect+              148
Intellect-               94
Name: count, dtype: int64

Exemplos de frases no grupo 'Intellect+':


,Frase
40,Admire a really clever scam.
104,Am able to come up with new and different ideas.
119,"Am able to see the ""big picture."""
136,Am always busy with something interesting.
154,Am an original thinker.



Exemplos de frases no grupo 'Agreeableness+':


,Frase
3,Accept people as they are.
4,Accept the consequences of my actions.
5,Accept the first thing that comes along.
6,Accept what others say.
9,Acknowledge others' accomplishments.


In [14]:
df_agrupado.to_csv('frases_agrupadas_personalidade.csv', index=False, encoding='utf-8')
print("Arquivo 'frases_agrupadas_personalidade.csv' exportado com sucesso!")

Arquivo 'frases_agrupadas_personalidade.csv' exportado com sucesso!


In [15]:
print("--- Top 5 frases por Traço (Maior Similaridade) ---\n")

for categoria in nomes_categorias:
    print(f"Traço: {categoria}")
    # Filtra pela categoria e ordena pelo score de similaridade
    top_frases = df_agrupado[df_agrupado['Categoria_Atribuida'] == categoria].sort_values(by='Score_Similaridade', ascending=False).head(5)

    if not top_frases.empty:
        for i, row in top_frases.iterrows():
            print(f"  - {row['Frase']} (Score: {row['Score_Similaridade']:.4f})")
    else:
        print("  (Nenhuma frase atribuída a este grupo)")
    print("-" * 30)

--- Top 5 frases por Traço (Maior Similaridade) ---

Traço: Surgency+
  - Like taking risks. (Score: 0.5801)
  - Am an energetic person. (Score: 0.5640)
  - Speak rapidly. (Score: 0.5616)
  - Take deviant positions. (Score: 0.5532)
  - Take risks. (Score: 0.5526)
------------------------------
Traço: Surgency-
  - Seek quiet. (Score: 0.6677)
  - Retreat from others. (Score: 0.6302)
  - Avoid eye contact. (Score: 0.5903)
  - Ammore of a loner than most people. (Score: 0.5784)
  - Rarely overindulge. (Score: 0.5758)
------------------------------
Traço: Agreeableness+
  - Reassure others. (Score: 0.6385)
  - Sense others' wishes. (Score: 0.6303)
  - Show my gratitude. (Score: 0.6184)
  - Care about others. (Score: 0.6150)
  - Like to help others. (Score: 0.6016)
------------------------------
Traço: Agreeableness-
  - Distrust people. (Score: 0.6040)
  - Try not to do favors for others. (Score: 0.5997)
  - Am upset by the misfortunes of strangers. (Score: 0.5905)
  - Try not to think abo